# exp02 — DiffMG α 튜닝

| 항목 | 값 |
|---|---|
| config | `experiments/configs/exp02_alpha_tuning.yaml` |
| 결과 저장 | `experiments/results/exp02_alpha_tuning/` |
| 변경점 | `lr_alpha` 0.005 → **0.02** / `diffmg_temperature` 1.0 → **0.5** |
| 목적 | exp01 α 균등(≈1/8) 문제 해결 — 관계 게이팅 실질 분화 유도 |
| 가설 | α_r 분산 증가 → 유의미한 관계(product_has_kw, ip_has_kw 등)에 가중치 집중 → PR-AUC 개선 |

In [ ]:
import os, sys
ROOT = os.path.abspath(os.path.join(os.path.dirname('__file__'), '..', '..'))
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)
os.chdir(ROOT)

import matplotlib
matplotlib.rcParams['font.family'] = 'Malgun Gothic'
matplotlib.rcParams['axes.unicode_minus'] = False

from experiments.exp_utils import (
    run_experiment, load_experiment, compare_experiments,
    plot_alpha_heatmap, plot_training_curve, print_metrics_table, print_recommendations
)

EXP_NAME = 'exp02_alpha_tuning'
CFG_PATH = 'experiments/configs/exp02_alpha_tuning.yaml'
print('ROOT:', ROOT)

## 1. 학습 실행

In [ ]:
results = run_experiment(CFG_PATH, EXP_NAME)

## 2. 성능 지표 (vs exp01)

In [ ]:
import pandas as pd
print_metrics_table(results)

# exp01 비교
try:
    r01 = load_experiment('exp01_baseline')
    print('\n[exp01 test]')
    print_metrics_table(r01)
except FileNotFoundError:
    print('exp01 결과 없음')

## 3. 학습 곡선

In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(9, 4))
plot_training_curve(results.get('history', []), ax=ax)
plt.savefig(f'experiments/results/{EXP_NAME}/training_curve.png', dpi=120, bbox_inches='tight')
plt.show()

## 4. DiffMG α_r — 관계 중요도 히트맵

> temperature=0.5 + lr_alpha=0.02 적용 후 α_r 분포 변화 확인.
> 균등에서 벗어나면 관계 게이팅 작동 중.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 3))
try:
    plot_alpha_heatmap('exp01_baseline', ax=axes[0])
    axes[0].set_title('exp01 (baseline, temperature=1.0)')
except Exception as e:
    axes[0].set_title(f'exp01 로드 실패: {e}')
plot_alpha_heatmap(EXP_NAME, ax=axes[1])
axes[1].set_title('exp02 (α tuned, temperature=0.5)')
plt.savefig(f'experiments/results/{EXP_NAME}/alpha_heatmap_compare.png', dpi=120, bbox_inches='tight')
plt.show()

## 5. 순회 추천 샘플

In [ ]:
print_recommendations(results)

## 6. 실험 간 비교

In [ ]:
df = compare_experiments(['exp01_baseline', 'exp02_alpha_tuning', 'exp03_complement_edges'])
display(df[['exp', 'val_pr_auc', 'val_auc_roc', 'test_pr_auc', 'test_auc_roc', 'test_f1']])

## 7. 분석 메모

아래 셀에 실험 결과 해석을 기록하세요.

### α_r 분화 여부
- [ ] 균등(≈1/8=0.125) 유지 → temperature 추가 하향 필요
- [ ] 일부 관계 집중 → 어떤 관계가 높았는가?

### PR-AUC 변화
- exp01 test PR-AUC: 0.4844
- exp02 test PR-AUC: __(실행 후 기록)__

### 다음 실험 방향
- α 분화 불충분 시: temperature 0.3 추가 시도
- PR-AUC 상승 확인 시: exp03 complement 엣지로 진행